In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_FACT_CUSTODIES_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    fact_custodies_df = session.sql(f"""
        SELECT
            c.CUS_ID,
            d.CUSTODIES_INFO_ID,
            c.TWENTY_NINE_C_IND,
            c.START_DATE,
            c.PERSON_PERSON_ID,
            c.ORGN_ORGN_ID,
            c.VPA_VPA_ID,
            c.SUR_SUR_ID,
            c.CAP_CAP_ID,
            c.POI_POI_ID,
            c.CHILD_ADJUDICATED_DATE,
            c.END_DATE,
            c.TWENTY_NINE_C_DATE,
            c.REASONABLE_EFFORT_DATE,
            c.PERSON_PERSON_CHILD_ID,
            c.MOD_CAP_CAP_ID,
            c.ADD_TS,
            c.ADD_USER,
            c.MOD_TS,
            c.MOD_USER,
            CURRENT_TIMESTAMP() AS CREATED_DATE,
            c.DELETED_DATE,
            c.CHECKSUM
        FROM {STG}.{STG_CUSTODIES} c
        LEFT JOIN {DWH}.{DIM_CUSTODIES_INFO} d
            ON c.LEGAL_STATUS_CODE IS NOT DISTINCT FROM d.LEGAL_STATUS_CODE_AV_ID
           AND c.END_REASON_CODE IS NOT DISTINCT FROM d.END_REASON_CODE_AV_ID
           AND c.CUSTODY_TYPE IS NOT DISTINCT FROM d.CUSTODY_TYPE_AV_ID
           AND d.UPDATED_DATE IS NULL
    """)

    fact_custodies_df.create_or_replace_temp_view("TEMP_FACT_CUSTODIES")

    merge_result = session.sql(f"""
        MERGE INTO {DWH}.{FACT_CUSTODIES} tgt
        USING TEMP_FACT_CUSTODIES src
            ON tgt.CUS_ID = src.CUS_ID
        WHEN MATCHED AND (
            src.CHECKSUM <> tgt.CHECKSUM
        ) THEN
            UPDATE SET
                CUSTODIES_INFO_ID = src.CUSTODIES_INFO_ID,
                TWENTY_NINE_C_IND = src.TWENTY_NINE_C_IND,
                START_DATE = src.START_DATE,
                PERSON_PERSON_ID = src.PERSON_PERSON_ID,
                ORGN_ORGN_ID = src.ORGN_ORGN_ID,
                VPA_VPA_ID = src.VPA_VPA_ID,
                SUR_SUR_ID = src.SUR_SUR_ID,
                CAP_CAP_ID = src.CAP_CAP_ID,
                POI_POI_ID = src.POI_POI_ID,
                CHILD_ADJUDICATED_DATE = src.CHILD_ADJUDICATED_DATE,
                END_DATE = src.END_DATE,
                TWENTY_NINE_C_DATE = src.TWENTY_NINE_C_DATE,
                REASONABLE_EFFORT_DATE = src.REASONABLE_EFFORT_DATE,
                PERSON_PERSON_CHILD_ID = src.PERSON_PERSON_CHILD_ID,
                MOD_CAP_CAP_ID = src.MOD_CAP_CAP_ID,
                ADD_TS = src.ADD_TS,
                ADD_USER = src.ADD_USER,
                MOD_TS = src.MOD_TS,
                MOD_USER = src.MOD_USER,
                DELETED_DATE = src.DELETED_DATE,
                CHECKSUM = src.CHECKSUM
        WHEN NOT MATCHED THEN
            INSERT (
                CUS_ID,
                CUSTODIES_INFO_ID,
                TWENTY_NINE_C_IND,
                START_DATE,
                PERSON_PERSON_ID,
                ORGN_ORGN_ID,
                VPA_VPA_ID,
                SUR_SUR_ID,
                CAP_CAP_ID,
                POI_POI_ID,
                CHILD_ADJUDICATED_DATE,
                END_DATE,
                TWENTY_NINE_C_DATE,
                REASONABLE_EFFORT_DATE,
                PERSON_PERSON_CHILD_ID,
                MOD_CAP_CAP_ID,
                ADD_TS,
                ADD_USER,
                MOD_TS,
                MOD_USER,
                CREATED_DATE,
                DELETED_DATE,
                CHECKSUM
            )
            VALUES (
                src.CUS_ID,
                src.CUSTODIES_INFO_ID,
                src.TWENTY_NINE_C_IND,
                src.START_DATE,
                src.PERSON_PERSON_ID,
                src.ORGN_ORGN_ID,
                src.VPA_VPA_ID,
                src.SUR_SUR_ID,
                src.CAP_CAP_ID,
                src.POI_POI_ID,
                src.CHILD_ADJUDICATED_DATE,
                src.END_DATE,
                src.TWENTY_NINE_C_DATE,
                src.REASONABLE_EFFORT_DATE,
                src.PERSON_PERSON_CHILD_ID,
                src.MOD_CAP_CAP_ID,
                src.ADD_TS,
                src.ADD_USER,
                src.MOD_TS,
                src.MOD_USER,
                src.CREATED_DATE,
                src.DELETED_DATE,
                src.CHECKSUM
            )
    """).collect()

    rows_inserted = merge_result[0][0]
    rows_updated = merge_result[0][1]
    rows_loaded = rows_inserted + rows_updated

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_CUSTODIES load completed. Rows Inserted: {rows_inserted}, Rows Updated: {rows_updated}"
    )

    print(f"DWH LOAD SUCCESS | Rows Inserted: {rows_inserted} | Rows Updated: {rows_updated}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_CUSTODIES load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED: {error_message}")